## Upgrading the Kalman Filter Model for RocketPy and Hardware
In previous iterations we simplified the model to be 1-dimensional. We will now introduce the missing two dimensions (x/y/z) to the model and then continue by modelling the attitude with gyro and magnetometer before doing a coordinate transformation.

### General Filter Configuration
The Kalman Filter proposed will still be of 4th order, since this yielded the best matching performance and also fits the theoretical model the best, since drag causes a change in acceleration which is modelled as jerk. For now the filter, as per its Transition matrix _**F**_ does not know the influence of velocity on drag. This shortcoming can be solved by linearizing the behaviour into a EKF but is for now just part of the process noise in the jerk which has to be chosen appropriately.

### The State
$$
\mathbf x =\begin{bmatrix} r_x \\ r_y \\ r_z \\ v_x \\ v_y \\ v_z \\ a_x \\ a_y \\ a_z \\ j_x \\ j_y \\ j_z   \end{bmatrix}
$$

### State Transition Matrix F
Since the state has dimension 12, the F matrix will be of shape 12x12. *dt* refers to the timestep which will most likely be $5ms$ in our case since the control software frequency goal is set at $200Hz$.
$$
\mathbf F =\begin{bmatrix}  
                            1 & 0 & 0 &     dt & 0 & 0 &    \frac{1}{2}dt^2 & 0 & 0      & \frac{1}{6}dt^3 & 0 & 0 \\
                            0 & 1 & 0 &     0 & dt & 0 &    0 & \frac{1}{2}dt^2 & 0      & 0 & \frac{1}{6}dt^3 & 0 \\
                            0 & 0 & 1 &     0 & 0 & dt &    0 & 0 & \frac{1}{2}dt^2      & 0 & 0 & \frac{1}{6}dt^3 \\


                            0 & 0 & 0 &     1 & 0 & 0   & dt & 0 & 0       & \frac{1}{2}dt^2  & 0 & 0 \\
                            0 & 0 & 0 &     0 & 1 & 0   & 0 & dt & 0       & 0 & \frac{1}{2}dt^2 & 0 \\
                            0 & 0 & 0 &     0 & 0 & 1   & 0 & 0 & dt       & 0 & 0 & \frac{1}{2}dt^2 \\

                            0 & 0 & 0 &     0 & 0 & 0 &    1 & 0 & 0   & dt & 0 & 0       \\
                            0 & 0 & 0 &     0 & 0 & 0 &    0 & 1 & 0   & 0 & dt & 0       \\
                            0 & 0 & 0 &     0 & 0 & 0 &    0 & 0 & 1   & 0 & 0 & dt       \\

                            0 & 0 & 0 &     0 & 0 & 0 &    0 & 0 & 0 &    1 & 0 & 0          \\
                            0 & 0 & 0 &     0 & 0 & 0 &    0 & 0 & 0 &    0 & 1 & 0          \\
                            0 & 0 & 0 &     0 & 0 & 0 &    0 & 0 & 0 &    0 & 0 & 1          \\
                           
\end{bmatrix}
$$

### Process Noise Matrix Q
We will go with the previous choice (from the 1D Model) of a discrete Wiener Process adapted to all 3 dimensions. [check this later](https://math.stackexchange.com/questions/881182/how-to-derive-the-process-noise-co-variance-matrix-q-in-this-kalman-filter-examp)

### Measurement Vector z and Matrices H
For the version running on the *STM32F4* we will most likely get asyncronous measurements from our sensors. Important for our position are the *primary IMU*, the *secondary IMU* and the *barometer*. Because of the asyncronous nature of incoming sensor events, I chose to have a measurement vector $z$, the conversion matrix $H$ and the noise matrix $R_k$ for all sensor types individually.

#### z, H and R for the IMU's
$z$ holds the acceleration values in all three axis, already converted to the **ENU** coordinate system using the inverse of the matrix describing the attitude and with gravity accounted for (subtracted away).
$$
\mathbf z_{primIMU} = \begin{bmatrix} a_x \\ a_y \\ a_z  \end{bmatrix}
$$

$$
H_{primIMu} =
\begin{bmatrix}
0 & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0 \\
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1 \\
0 & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0
\end{bmatrix}
$$

$H$ can later also be equipped with the cross-axis-sensitivity obtained from the datasheet or own measurements. This would lead us to have two separate ones for each sensor though.
We will have two separate $R_k$ observation noise matrices to capture the precision of each sensor and the appropriate one will be used.

#### z, H and R for the Barometer
$z$ holds the $r_z$ value already in the **ENU** coordinate system.

$$
\mathbf z_{baro} = \begin{bmatrix} r_z \end{bmatrix}
$$

$$
\mathbf H_{baro} =\begin{bmatrix} 
0 \\
0 \\
1 \\
0 \\
0 \\
0 \\
0 \\
0 \\
0 \\
0 \\
0 \\
0
  \end{bmatrix}
$$



Here $R_k$ will be one dimensional and only hold the variance of 


### Sensor Failure Detection and Reactivation procedure
- outlier detection with state and others
a) fault in underlying protocol (eg. I2C/SPI hangs)
b) 

### Ressources
- [Attitude Representation](https://link.springer.com/content/pdf/10.1007/978-3-030-04275-2_4)

In [1]:
import numpy as np